# BUILD-01C — Glacier Inventory Ingestion

This notebook is the first executable HG-SCRIS ingestion workflow. It is designed for Google Colab and uses an explicit source path rather than silently downloading or redistributing large external datasets.

Pipeline: source → metadata → read → schema inspection → geometry/CRS QA → standardized output → provenance.

In [ ]:
!pip -q install geopandas pyogrio shapely pyproj pandas pyyaml

from pathlib import Path
import geopandas as gpd
import pandas as pd

SOURCE_PATH = Path('/content/rgi_source.shp')  # Set to the downloaded RGI source file
OUTPUT_DIR = Path('/content/hgscris_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE_PATH)
print('Output:', OUTPUT_DIR)

In [ ]:
if not SOURCE_PATH.exists():
    raise FileNotFoundError('Place the approved RGI source file at SOURCE_PATH before running ingestion.')
gdf = gpd.read_file(SOURCE_PATH)
print(gdf.shape)
print(gdf.crs)
display(gdf.head())

In [ ]:
# Basic integrity checks
if gdf.crs is None:
    raise ValueError('Input glacier layer has no CRS.')
invalid_count = (~gdf.geometry.is_valid).sum()
empty_count = gdf.geometry.is_empty.sum()
print({'invalid_geometry': int(invalid_count), 'empty_geometry': int(empty_count), 'crs': str(gdf.crs)})
if invalid_count or empty_count:
    raise ValueError('Geometry QA failed; inspect source before promotion.')

In [ ]:
# Preserve source attributes; only normalize field labels at this stage.
gdf.columns = [str(c).strip().lower() for c in gdf.columns]
out_path = OUTPUT_DIR / 'glacier_inventory_standardized.gpkg'
gdf.to_file(out_path, layer='glaciers', driver='GPKG')
print('Wrote:', out_path)